In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
BASE = "/content/drive/MyDrive/Thesis/Analysis/Data/thesis"

In [ ]:
FILES = {
    "A": f"{BASE}/Ver A.csv",
    "B": f"{BASE}/Ver B.csv",
    "C": f"{BASE}/Ver C.csv",
    "D": f"{BASE}/Ver D.csv",
}

In [ ]:
ENGLISH_VERSION = "B"

In [ ]:
OUT_LONG = f"{BASE}/thesis_long_format.csv"
OUT_RESP = f"{BASE}/thesis_respondent_level.csv"

In [ ]:
COLUMN_NAMES = [
    "timestamp",
    "live_tw",
    "vig1_choice",
    "vig2_choice",
    "vig3_choice",
    "vig4_choice",
    "p_trust1",      # platform: dispute resolution
    "p_trust2",      # platform: data privacy
    "p_trust3",      # platform: safe shopping
    "s_trust1",      # seller: trustworthy
    "s_trust2",      # seller: delivers as promised
    "c_trust1",      # consumer: clothing meets expectations
    "c_trust2",      # consumer: past experiences positive
    "g_trust1",      # general: trust other people
    "risk_aversion", # risk aversion single item
    "age",
    "gender",
    "status",
    "income",
    "cloth_budget",
]

In [ ]:
TRUST_ITEMS = [
    "p_trust1", "p_trust2", "p_trust3",
    "s_trust1", "s_trust2",
    "c_trust1", "c_trust2",
    "g_trust1",
]

In [ ]:
VIG_COLS = ["vig1_choice", "vig2_choice", "vig3_choice", "vig4_choice"]

In [ ]:
DEMO_COLS = ["age_cat", "gender_cat", "status_cat", "income_cat", "budget_cat"]

##Maping

In [ ]:
VIGNETTE_MAP = {
    "Buy online": 1,
    "Buy in-store": 0,
    "線上購買": 1,
    "到實體店購買": 0,
}

In [ ]:
LIVE_TW_MAP = {
    "Yes": 1, "No": 0,
    "是": 1, "否": 0,
}

In [ ]:
GENDER_MAP = {
    "Male": 1, "Female": 2, "Other": 3, "Prefer not to say": 4,
    "男性": 1, "女性": 2, "其他": 3, "不願透露": 4,
}

In [ ]:
STATUS_MAP = {
    "Student": 1, "Working": 2, "Other": 3,
    "學生": 1,
    "工作中（全職或兼職）": 2,
    "工作中 (全職或兼職)": 2,
    "其他": 3,
}

In [ ]:
AGE_MAP = {
    "18-24": 1, "18–24": 1,
    "25-34": 2, "25–34": 2,
    "35-44": 3, "35–44": 3,
    "45-54": 4, "45–54": 4,
    "54+": 5, "55+": 5,
    "18-24 歲": 1, "18–24 歲": 1,
    "25-34 歲": 2, "25–34 歲": 2,
    "35-44 歲": 3, "35–44 歲": 3,
    "45-54 歲": 4, "45–54 歲": 4,
    "54歲以上": 5, "55 以上": 5, "45 歲以上": 4,
}

In [ ]:
INCOME_MAP = {
    "None": 0, "無收入": 0,
    "< NT$20,000": 1, "<NT$20,000": 1, "< NT$20,000 ": 1,"< 20,000": 1,
"<20,000": 1,
"< 20,000 ": 1,
    "20–39,999": 2, "20-39,999": 2,
    "40–59,999": 3, "40-59,999": 3,
    "60–79,999": 4, "60-79,999": 4,
    "80–99,999": 5, "80-99,999": 5,
    "≥100,000": 6, ">=100,000": 6,
    "Prefer not to say": 7, "不願透露": 7,
}

In [ ]:
BUDGET_MAP = {
    "<1,000": 1, "＜1,000": 1,
    "1,000–2,999": 2, "1,000-2,999": 2,
    "3,000–4,999": 3, "3,000-4,999": 3,
    "5,000–9,999": 4, "5,000-9,999": 4,
    "≥10,000": 5, ">=10,000": 5,
    "Prefer not to say": 6, "不願透露": 6,
}

##Vignette maping

In [ ]:
DESIGN_MAP = pd.DataFrame([
    # Low reputation, Low price (700 NTD), discount 0/10/25/50
    {"version": "A", "vignette_num": 1, "vignette_id": 1,  "reputation_high": 0, "price_high": 0, "discount": 0},
    {"version": "B", "vignette_num": 2, "vignette_id": 2,  "reputation_high": 0, "price_high": 0, "discount": 10},
    {"version": "C", "vignette_num": 3, "vignette_id": 3,  "reputation_high": 0, "price_high": 0, "discount": 25},
    {"version": "D", "vignette_num": 4, "vignette_id": 4,  "reputation_high": 0, "price_high": 0, "discount": 50},

    # Low reputation, High price (1500 NTD), discount 0/10/25/50
    {"version": "B", "vignette_num": 1, "vignette_id": 5,  "reputation_high": 0, "price_high": 1, "discount": 0},
    {"version": "A", "vignette_num": 2, "vignette_id": 6,  "reputation_high": 0, "price_high": 1, "discount": 10},
    {"version": "D", "vignette_num": 3, "vignette_id": 7,  "reputation_high": 0, "price_high": 1, "discount": 25},
    {"version": "C", "vignette_num": 4, "vignette_id": 8,  "reputation_high": 0, "price_high": 1, "discount": 50},

    # High reputation, Low price (700 NTD), discount 0/10/25/50
    {"version": "C", "vignette_num": 1, "vignette_id": 9,  "reputation_high": 1, "price_high": 0, "discount": 0},
    {"version": "D", "vignette_num": 2, "vignette_id": 10, "reputation_high": 1, "price_high": 0, "discount": 10},
    {"version": "A", "vignette_num": 3, "vignette_id": 11, "reputation_high": 1, "price_high": 0, "discount": 25},
    {"version": "B", "vignette_num": 4, "vignette_id": 12, "reputation_high": 1, "price_high": 0, "discount": 50},

    # High reputation, High price (1500 NTD), discount 0/10/25/50
    {"version": "D", "vignette_num": 1, "vignette_id": 13, "reputation_high": 1, "price_high": 1, "discount": 0},
    {"version": "C", "vignette_num": 2, "vignette_id": 14, "reputation_high": 1, "price_high": 1, "discount": 10},
    {"version": "B", "vignette_num": 3, "vignette_id": 15, "reputation_high": 1, "price_high": 1, "discount": 25},
    {"version": "A", "vignette_num": 4, "vignette_id": 16, "reputation_high": 1, "price_high": 1, "discount": 50},
])

##Functions

In [ ]:
def clean_str(series):
    """Convert series to string and strip whitespace."""
    return series.astype("string").str.strip()

In [ ]:
def load_version(filepath, version_label):
    """Load one CSV, rename columns positionally, add version label."""
    df = pd.read_csv(filepath, keep_default_na=False, na_values=[""])
    df.columns = COLUMN_NAMES
    df["version"] = version_label
    df = df.drop(columns=["timestamp"])
    print(f"  Version {version_label}: {len(df)} responses loaded")
    return df

In [ ]:
def recode_demographics(df):
    """Recode all demographic variables to numeric categories."""
    df = df.copy()
    df["live_tw"]     = clean_str(df["live_tw"]).map(LIVE_TW_MAP).astype("Int64")
    df["gender_cat"]  = clean_str(df["gender"]).map(GENDER_MAP).astype("Int64")
    df["status_cat"]  = clean_str(df["status"]).map(STATUS_MAP).astype("Int64")
    df["age_cat"]     = clean_str(df["age"]).map(AGE_MAP).astype("Int64")
    df["income_cat"]  = clean_str(df["income"]).map(INCOME_MAP).astype("Int64")
    df["budget_cat"]  = clean_str(df["cloth_budget"]).map(BUDGET_MAP).astype("Int64")
    return df

In [ ]:
def recode_vignettes(df):
    """Recode vignette choices to binary (1=online, 0=in-store)."""
    df = df.copy()
    for col in VIG_COLS:
        df[col] = clean_str(df[col]).map(VIGNETTE_MAP).astype("Int64")
    return df

In [ ]:
def ensure_numeric_scales(df):
    """Ensure trust items and risk aversion are numeric."""
    df = df.copy()
    for col in TRUST_ITEMS + ["risk_aversion"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    return df

In [ ]:
def filter_taiwan_residents(df):
    """Keep only respondents who live in Taiwan."""
    before = len(df)
    df = df[df["live_tw"] == 1].copy()
    dropped = before - len(df)
    if dropped > 0:
        print(f"    Dropped {dropped} non-Taiwan residents")
    return df

In [ ]:
def check_unmapped_values(df):
    """Print any values that didn't map correctly."""
    cols_to_check = {
        "gender_cat": "gender",
        "status_cat": "status",
        "age_cat": "age",
        "income_cat": "income",
        "budget_cat": "cloth_budget",
    }
    issues = False
    for coded, raw in cols_to_check.items():
        mask = df[coded].isna()
        if mask.any():
            issues = True
            print(f"    WARNING: {mask.sum()} unmapped values in {raw}:")
            print(f"      {df.loc[mask, raw].unique()}")
    for col in VIG_COLS:
        mask = df[col].isna()
        if mask.any():
            issues = True
            print(f"    WARNING: {mask.sum()} unmapped values in {col}")
    if not issues:
        print(f"    All values mapped successfully")

In [ ]:
def build_respondent_df(dfs_clean):
    """Combine all versions, add survey_id."""
    df = pd.concat(dfs_clean.values(), ignore_index=True)
    df["survey_id"] = range(1, len(df) + 1)
    return df

In [ ]:
def build_long_format(df_resp):
    """Melt to long format, merge experimental conditions and respondent traits."""
    df_long = df_resp.melt(
        id_vars=["survey_id", "version"],
        value_vars=VIG_COLS,
        var_name="vig_col",
        value_name="online_choice",
    )
    df_long["vignette_num"] = df_long["vig_col"].str.extract(r"(\d+)").astype(int)
    df_long = df_long.drop(columns=["vig_col"])

    # merge experimental conditions
    df_long = df_long.merge(DESIGN_MAP, on=["version", "vignette_num"], how="left")

    # merge respondent-level variables
    resp_cols = (
        ["survey_id", "version"]
        + TRUST_ITEMS
        + ["risk_aversion"]
        + ["age_cat", "gender_cat", "status_cat", "income_cat", "budget_cat"]
    )
    df_long = df_long.merge(df_resp[resp_cols], on=["survey_id", "version"], how="left")

    return df_long

##Main

In [ ]:
dfs_clean = {}
for version, filepath in FILES.items():
    df = load_version(filepath, version)
    df = recode_demographics(df)
    df = recode_vignettes(df)
    df = ensure_numeric_scales(df)
    df = filter_taiwan_residents(df)
    check_unmapped_values(df)
    dfs_clean[version] = df

  Version A: 31 responses loaded
    All values mapped successfully
  Version B: 25 responses loaded
    All values mapped successfully
  Version C: 34 responses loaded
    Dropped 1 non-Taiwan residents
    All values mapped successfully
  Version D: 32 responses loaded
    All values mapped successfully


In [ ]:
df_resp = build_respondent_df(dfs_clean)
print(f"\nTotal respondents: {len(df_resp)}")


Total respondents: 121


In [ ]:
df_long = build_long_format(df_resp)
print(f"Long format: {df_long.shape[0]} observations ({df_long.shape[0] // 4} respondents × 4 vignettes)")

Long format: 484 observations (121 respondents × 4 vignettes)


In [ ]:
print(f"\nSanity checks:")
print(f"  Observations per respondent: {df_long.groupby('survey_id').size().unique()}")
print(f"  Versions in data: {sorted(df_long['version'].unique())}")
print(f"  Discount levels: {sorted(df_long['discount'].unique())}")
print(f"  Missing online_choice: {df_long['online_choice'].isna().sum()}")



Sanity checks:
  Observations per respondent: [4]
  Versions in data: ['A', 'B', 'C', 'D']
  Discount levels: [np.int64(0), np.int64(10), np.int64(25), np.int64(50)]
  Missing online_choice: 0


##export

In [ ]:
df_long.to_csv(OUT_LONG, index=False)
print(f"\nExported: {OUT_LONG}")


Exported: /content/drive/MyDrive/Thesis/Analysis/Data/thesis/thesis_long_format.csv


In [ ]:
df_resp.to_csv(OUT_RESP, index=False)
print(f"Exported: {OUT_RESP}")

Exported: /content/drive/MyDrive/Thesis/Analysis/Data/thesis/thesis_respondent_level.csv
